> <p><small><small>Copyright 2023 DeepMind Technologies Limited.</small></p>
> <p><small><small>Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at <a href="http://www.apache.org/licenses/LICENSE-2.0">http://www.apache.org/licenses/LICENSE-2.0</a>.</small></small></p>
> <p><small><small>Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.</small></small></p>

# Installation and Initialization


In [1]:
# @title Pip install graphcast and dependencies

%pip install -q --upgrade https://github.com/deepmind/graphcast/archive/master.zip
%pip install -q jax[cuda12] optax #jax[tpu]==0.6.2

     | 1.7 MB 7.3 MB/s 0:00:000m
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 105.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.0/101.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.3/172.3 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.3/374.3 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.5/307.5 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 90.8 MB/s eta 0:00:00ta 0:00:01
ERROR: pip's dependency re

In [2]:
# @title Patch installed graphcast.py with freeze_encoder / freeze_processor support
# Applies precise string patches to the pip-installed graphcast.py already on
# the VM — no file upload, no Google Drive, no internet required.
# Each patch is individually idempotent: applied only if its target is absent.
# Patches 3 & 4 also handle upgrading a previous patch that lacked comments.

import importlib, inspect
from pathlib import Path

_DEST = Path('/usr/local/lib/python3.12/dist-packages/graphcast/graphcast.py')
_src  = _DEST.read_text(encoding='utf-8')
_changed = False

# ── Patch 0: ensure `import jax` is present ───────────────────────────────────
# graphcast.py uses jnp throughout but the pip version has no bare `import jax`.
# Our stop_gradient calls need jax.lax, so insert it before `import jax.numpy`.
if 'import jax\n' not in _src:
    _ANCHOR = 'import jax.numpy as jnp'
    assert _ANCHOR in _src, (
        "PATCH 0 FAILED — 'import jax.numpy as jnp' not found.\n"
        "Unexpected file layout in:\n  " + str(_DEST))
    _src = _src.replace(_ANCHOR, 'import jax\n' + _ANCHOR, 1)
    print("Patch 0 applied : added 'import jax'")
    _changed = True
else:
    print("Patch 0 skipped : 'import jax' already present")

# ── Patch 1: __init__ signature ───────────────────────────────────────────────
_OLD1 = '  def __init__(self, model_config: ModelConfig, task_config: TaskConfig):'
_NEW1 = ('  def __init__(self, model_config: ModelConfig, task_config: TaskConfig,\n'
         '               freeze_encoder: bool = False, freeze_processor: bool = False):')
if _OLD1 in _src:
    _src = _src.replace(_OLD1, _NEW1, 1)
    print("Patch 1 applied : freeze flags added to __init__ signature")
    _changed = True
elif _NEW1 in _src:
    print("Patch 1 skipped : __init__ signature already patched")
else:
    raise AssertionError(
        "PATCH 1 FAILED — __init__ signature not found in original or patched form.\n"
        "The pip-installed graphcast version may have changed. Check:\n  " + str(_DEST))

# ── Patch 2: __init__ body — store flags as instance attributes ───────────────
_OLD2 = ('    """Initializes the predictor."""\n'
         '    self._spatial_features_kwargs')
_NEW2 = ('    """Initializes the predictor."""\n'
         '    self._freeze_encoder = freeze_encoder\n'
         '    self._freeze_processor = freeze_processor\n'
         '    self._spatial_features_kwargs')
if _OLD2 in _src:
    _src = _src.replace(_OLD2, _NEW2, 1)
    print("Patch 2 applied : freeze attributes stored in __init__ body")
    _changed = True
elif '_freeze_encoder = freeze_encoder' in _src:
    print("Patch 2 skipped : freeze attributes already in __init__ body")
else:
    raise AssertionError(
        "PATCH 2 FAILED — __init__ body anchor not found.\n"
        "Expected '\"\"\"Initializes the predictor.\"\"\"' followed by "
        "'self._spatial_features_kwargs'.")

# ── Patch 3: stop_gradient after encoder output ───────────────────────────────
# _NEW3 is the exact text in the local modified graphcast.py (with comments).
# _OLD3_NO_COMMENT handles upgrading a prior run that applied the if-block
# without the explanatory comment lines.
_OLD3 = ('     ) = self._run_grid2mesh_gnn(grid_node_features)\n\n'
         '    # Run message passing in the multimesh.')
_OLD3_NO_COMMENT = ('     ) = self._run_grid2mesh_gnn(grid_node_features)\n\n'
                    '    if self._freeze_encoder:\n'
                    '      latent_mesh_nodes = jax.lax.stop_gradient(latent_mesh_nodes)\n'
                    '      latent_grid_nodes = jax.lax.stop_gradient(latent_grid_nodes)\n\n'
                    '    # Run message passing in the multimesh.')
_NEW3 = ('     ) = self._run_grid2mesh_gnn(grid_node_features)\n\n'
         '    # Stop gradient through encoder outputs to save HBM memory during backprop.\n'
         '    # This prevents JAX from storing encoder activations for gradient computation.\n'
         '    if self._freeze_encoder:\n'
         '      latent_mesh_nodes = jax.lax.stop_gradient(latent_mesh_nodes)\n'
         '      latent_grid_nodes = jax.lax.stop_gradient(latent_grid_nodes)\n\n'
         '    # Run message passing in the multimesh.')
if _OLD3 in _src:
    _src = _src.replace(_OLD3, _NEW3, 1)
    print("Patch 3 applied : stop_gradient + comments after encoder")
    _changed = True
elif '# Stop gradient through encoder outputs' in _src:
    print("Patch 3 skipped : encoder stop_gradient (with comments) already present")
elif _OLD3_NO_COMMENT in _src:
    _src = _src.replace(_OLD3_NO_COMMENT, _NEW3, 1)
    print("Patch 3 upgraded: added missing comments before encoder stop_gradient")
    _changed = True
else:
    raise AssertionError(
        "PATCH 3 FAILED — encoder output anchor not found in any expected form.\n"
        "Expected '_run_grid2mesh_gnn' followed by '# Run message passing'.")

# ── Patch 4: stop_gradient after processor output ─────────────────────────────
# Same strategy as Patch 3: full patch with comments + upgrade path for
# a previous run that applied the if-block without the comment lines.
_OLD4 = ('    updated_latent_mesh_nodes = self._run_mesh_gnn(latent_mesh_nodes)\n\n'
         '    # Transfer data frome the mesh to the grid.')
_OLD4_NO_COMMENT = ('    updated_latent_mesh_nodes = self._run_mesh_gnn(latent_mesh_nodes)\n\n'
                    '    if self._freeze_processor:\n'
                    '      updated_latent_mesh_nodes = jax.lax.stop_gradient(updated_latent_mesh_nodes)\n\n'
                    '    # Transfer data frome the mesh to the grid.')
_NEW4 = ('    updated_latent_mesh_nodes = self._run_mesh_gnn(latent_mesh_nodes)\n\n'
         '    # Stop gradient through processor outputs to save HBM memory during backprop.\n'
         '    # This prevents JAX from storing processor activations (16 steps) for gradients.\n'
         '    if self._freeze_processor:\n'
         '      updated_latent_mesh_nodes = jax.lax.stop_gradient(updated_latent_mesh_nodes)\n\n'
         '    # Transfer data frome the mesh to the grid.')
if _OLD4 in _src:
    _src = _src.replace(_OLD4, _NEW4, 1)
    print("Patch 4 applied : stop_gradient + comments after processor")
    _changed = True
elif '# Stop gradient through processor outputs' in _src:
    print("Patch 4 skipped : processor stop_gradient (with comments) already present")
elif _OLD4_NO_COMMENT in _src:
    _src = _src.replace(_OLD4_NO_COMMENT, _NEW4, 1)
    print("Patch 4 upgraded: added missing comments before processor stop_gradient")
    _changed = True
else:
    raise AssertionError(
        "PATCH 4 FAILED — processor output anchor not found in any expected form.\n"
        "Expected '_run_mesh_gnn' followed by '# Transfer data frome'.")

# ── Write only if something changed ───────────────────────────────────────────
if _changed:
    _DEST.write_text(_src, encoding='utf-8')
    print(f"\nWritten : {_DEST}  ({_DEST.stat().st_size:,} bytes)")
else:
    print("\nAll patches already present — file unchanged.")

# ── Reload and verify ─────────────────────────────────────────────────────────
from graphcast import graphcast as _gc_module
importlib.reload(_gc_module)

_sig = inspect.signature(_gc_module.GraphCast.__init__)
assert 'freeze_encoder'  in _sig.parameters, "freeze_encoder missing after patch"
assert 'freeze_processor' in _sig.parameters, "freeze_processor missing after patch"
print(f"\ngraphcast.py active.  GraphCast.__init__{_sig}")

Patch 0 applied : added 'import jax'
Patch 1 applied : freeze flags added to __init__ signature
Patch 2 applied : freeze attributes stored in __init__ body
Patch 3 applied : stop_gradient + comments after encoder
Patch 4 applied : stop_gradient + comments after processor

Written : /usr/local/lib/python3.12/dist-packages/graphcast/graphcast.py  (32,264 bytes)

graphcast.py active.  GraphCast.__init__(self, model_config: graphcast.graphcast.ModelConfig, task_config: graphcast.graphcast.TaskConfig, freeze_encoder: bool = False, freeze_processor: bool = False)


In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
import sys
for path in sys.path:
    print(path)

/content
/env/python
/usr/lib/python312.zip
/usr/lib/python3.12
/usr/lib/python3.12/lib-dynload

/usr/local/lib/python3.12/dist-packages
/usr/lib/python3/dist-packages
/usr/local/lib/python3.12/dist-packages/IPython/extensions
/root/.ipython


In [5]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

In [5]:
import dataclasses
import datetime
import functools
import math
import re
from typing import Optional, Mapping, Any

from google.cloud import storage
from graphcast import autoregressive
from graphcast import casting
from graphcast import checkpoint
from graphcast import data_utils
from graphcast import graphcast
from graphcast import normalization
from graphcast import rollout
from graphcast import xarray_jax
from graphcast import xarray_tree
from IPython.display import HTML
import ipywidgets as widgets
import haiku as hk
import jax
import jax.numpy as jnp

import numpy as np
import xarray

In [6]:
# @title Authenticate with Google Cloud Storage

gcs_client = storage.Client.create_anonymous_client()
gcs_bucket = gcs_client.get_bucket("dm_graphcast")
dir_prefix = "graphcast/"

def parse_file_parts(file_name):
  return dict(part.split("-", 1) for part in file_name.split("_"))

# Load the Data and initialize the model

## Load the model params

Choose one of the two ways of getting model params:
- **random**: You'll get random predictions, but you can change the model architecture, which may run faster or fit on your device.
- **checkpoint**: You'll get sensible predictions, but are limited to the model architecture that it was trained with, which may not fit on your device. In particular generating gradients uses a lot of memory, so you'll need at least 25GB of ram (TPUv4 or A100).

Checkpoints vary across a few axes:
- The mesh size specifies the internal graph representation of the earth. Smaller meshes will run faster but will have worse outputs. The mesh size does not affect the number of parameters of the model.
- The resolution and number of pressure levels must match the data. Lower resolution and fewer levels will run a bit faster. Data resolution only affects the encoder/decoder.
- All our models predict precipitation. However, ERA5 includes precipitation, while HRES does not. Our models marked as "ERA5" take precipitation as input and expect ERA5 data as input, while model marked "ERA5-HRES" do not take precipitation as input and are specifically trained to take HRES-fc0 as input (see the data section below).

We provide three pre-trained models.
1. `GraphCast`, the high-resolution model used in the GraphCast paper (0.25 degree resolution, 37 pressure levels), trained on ERA5 data from 1979 to 2017,

2. `GraphCast_small`, a smaller, low-resolution version of GraphCast (1 degree resolution, 13 pressure levels, and a smaller mesh), trained on ERA5 data from 1979 to 2015, useful to run a model with lower memory and compute constraints,

3. `GraphCast_operational`, a high-resolution model (0.25 degree resolution, 13 pressure levels) pre-trained on ERA5 data from 1979 to 2017 and fine-tuned on HRES data from 2016 to 2021. This model can be initialized from HRES data (does not require precipitation inputs).


In [7]:
# @title Load the model

source = "Checkpoint" # source_tab.get_title(source_tab.selected_index)
params_file = str('GraphCast_operational - ERA5-HRES 1979-2021 - resolution 0.25 - pressure levels 13 - mesh 2to6 - precipitation output only.npz')

if source == "Random":
  params = None  # Filled in below
  state = {}
  model_config = graphcast.ModelConfig(
      resolution=0,
      mesh_size=random_mesh_size.value,
      latent_size=random_latent_size.value,
      gnn_msg_steps=random_gnn_msg_steps.value,
      hidden_layers=1,
      radius_query_fraction_edge_length=0.6)
  task_config = graphcast.TaskConfig(
      input_variables=graphcast.TASK.input_variables,
      target_variables=graphcast.TASK.target_variables,
      forcing_variables=graphcast.TASK.forcing_variables,
      pressure_levels=graphcast.PRESSURE_LEVELS[random_levels.value],
      input_duration=graphcast.TASK.input_duration,
  )
else:
  assert source == "Checkpoint"
  with gcs_bucket.blob(f"{dir_prefix}params/{params_file}").open("rb") as f:
    ckpt = checkpoint.load(f, graphcast.CheckPoint)
  params = ckpt.params
  state = {}

  model_config = ckpt.model_config
  task_config = ckpt.task_config
  print("Model description:\n", ckpt.description, "\n")
  print("Model license:\n", ckpt.license, "\n")

model_config

Model description:
 
GraphCast model at 0.25deg resolution, with 13 pressure levels. This model is
trained on ERA5 data from 1979 to 2017, and fine-tuned on HRES-fc0 data from
2016 to 2021 and can be causally evaluated on 2022 and later years. This model
does not take `total_precipitation_6hr` as inputs and can make predictions in an
operational setting (i.e., initialised from HRES-fc0).
 

Model license:
 
The model weights are licensed under the Creative Commons
Attribution-NonCommercial-ShareAlike 4.0 International (CC BY-NC-SA 4.0). You
may obtain a copy of the License at:
https://creativecommons.org/licenses/by-nc-sa/4.0/.
The weights were trained on ERA5 data, see README for attribution statement.
 



ModelConfig(resolution=0.25, mesh_size=6, latent_size=512, gnn_msg_steps=16, hidden_layers=1, radius_query_fraction_edge_length=0.5999912857713345, mesh2grid_edge_normalization_factor=0.6180338738074472)

## Load the example data

Several example datasets are available, varying across a few axes:
- **Source**: fake, era5, hres
- **Resolution**: 0.25deg, 1deg, 6deg
- **Levels**: 13, 37
- **Steps**: How many timesteps are included

Not all combinations are available.
- Higher resolution is only available for fewer steps due to the memory requirements of loading them.
- HRES is only available in 0.25 deg, with 13 pressure levels.

The data resolution must match the model that is loaded.

Some transformations were done from the base datasets:
- We accumulated precipitation over 6 hours instead of the default 1 hour.
- For HRES data, each time step corresponds to the HRES forecast at leadtime 0, essentially providing an "initialisation" from HRES. See HRES-fc0 in the GraphCast paper for further description. Note that a 6h accumulation of precipitation is not available from HRES, so our model taking HRES inputs does not depend on precipitation. However, because our models predict precipitation, we include the ERA5 precipitation in the example data so it can serve as an illustrative example of ground truth.
- We include ERA5 `toa_incident_solar_radiation` in the data. Our model uses the radiation at -6h, 0h and +6h as a forcing term for each 1-step prediction. If the radiation is missing from the data (e.g. in an operational setting), it will be computed using a custom implementation that produces values similar to those in ERA5.

In [8]:
# @title Get and filter the list of available example datasets

dataset_file_options = [
    name for blob in gcs_bucket.list_blobs(prefix=dir_prefix+"dataset/")
    if (name := blob.name.removeprefix(dir_prefix+"dataset/"))]  # Drop empty string.

def data_valid_for_model(
    file_name: str, model_config: graphcast.ModelConfig, task_config: graphcast.TaskConfig):
  file_parts = parse_file_parts(file_name.removesuffix(".nc"))
  return (
      model_config.resolution in (0, float(file_parts["res"])) and
      len(task_config.pressure_levels) == int(file_parts["levels"]) and
      (
          ("total_precipitation_6hr" in task_config.input_variables and
           file_parts["source"] in ("era5", "fake")) or
          ("total_precipitation_6hr" not in task_config.input_variables and
           file_parts["source"] in ("hres", "fake"))
      )
  )


dataset_file = widgets.Dropdown(
    options=[
        (", ".join([f"{k}: {v}" for k, v in parse_file_parts(option.removesuffix(".nc")).items()]), option)
        for option in dataset_file_options
        if data_valid_for_model(option, model_config, task_config)
    ],
    description="Dataset file:",
    layout={"width": "max-content"})
widgets.VBox([
    dataset_file,
    widgets.Label(value="Run the next cell to load the dataset. Rerunning this cell clears your selection and refilters the datasets that match your model.")
])

In [9]:
# @title Load weather data

if not data_valid_for_model(dataset_file.value, model_config, task_config):
  raise ValueError(
      "Invalid dataset file, rerun the cell above and choose a valid dataset file.")

with gcs_bucket.blob(f"{dir_prefix}dataset/{dataset_file.value}").open("rb") as f:
  example_batch = xarray.load_dataset(f).compute()

assert example_batch.dims["time"] >= 3  # 2 for input, >=1 for targets

print(", ".join([f"{k}: {v}" for k, v in parse_file_parts(dataset_file.value.removesuffix(".nc")).items()]))

example_batch

source: hres, date: 2022-01-01, res: 0.25, levels: 13, steps: 12


<xarray.Dataset> Size: 5GB
Dimensions:                       (lat: 721, lon: 1440, batch: 1, time: 14,
                                   level: 13)
Coordinates:
  * lat                           (lat) float32 3kB -90.0 -89.75 ... 89.75 90.0
  * lon                           (lon) float32 6kB 0.0 0.25 0.5 ... 359.5 359.8
  * time                          (time) timedelta64[ns] 112B 0 days 00:00:00...
  * level                         (level) int32 52B 50 100 150 ... 850 925 1000
    datetime                      (batch, time) datetime64[ns] 112B 2022-01-0...
Dimensions without coordinates: batch
Data variables: (12/14)
    geopotential_at_surface       (lat, lon) float32 4MB 2.735e+04 ... -0.07617
    land_sea_mask                 (lat, lon) float32 4MB 1.0 1.0 1.0 ... 0.0 0.0
    2m_temperature                (batch, time, lat, lon) float32 58MB 250.3 ...
    mean_sea_level_pressure       (batch, time, lat, lon) float32 58MB 9.936e...
    10m_v_component_of_wind       (batch, time, lat, lon) float32 58MB -0.474...
    10m_u_component_of_wind       (batch, time, lat, lon) float32 58MB -5.817...
    ...                            ...
    temperature                   (batch, time, level, lat, lon) float32 756MB ...
    geopotential                  (batch, time, level, lat, lon) float32 756MB ...
    u_component_of_wind           (batch, time, level, lat, lon) float32 756MB ...
    v_component_of_wind           (batch, time, level, lat, lon) float32 756MB ...
    vertical_velocity             (batch, time, level, lat, lon) float32 756MB ...
    specific_humidity             (batch, time, level, lat, lon) float32 756MB ...

In [10]:
# @title Choose training and eval data to extract
train_steps = widgets.IntSlider(
    value=1, min=1, max=example_batch.sizes["time"]-2, description="Train steps")
eval_steps = widgets.IntSlider(
    value=example_batch.sizes["time"]-2, min=1, max=example_batch.sizes["time"]-2, description="Eval steps")

widgets.VBox([
    train_steps,
    eval_steps,
    widgets.Label(value="Run the next cell to extract the data. Rerunning this cell clears your selection.")
])

In [13]:
# @title Extract training and eval data

train_inputs, train_targets, train_forcings = data_utils.extract_inputs_targets_forcings(
    example_batch, target_lead_times=slice("6h", f"{train_steps.value*6}h"),
    **dataclasses.asdict(task_config))

eval_inputs, eval_targets, eval_forcings = data_utils.extract_inputs_targets_forcings(
    example_batch, target_lead_times=slice("6h", f"{eval_steps.value*6}h"),
    **dataclasses.asdict(task_config))

print("All Examples:  ", example_batch.dims.mapping)
print("Train Inputs:  ", train_inputs.dims.mapping)
print("Train Targets: ", train_targets.dims.mapping)
print("Train Forcings:", train_forcings.dims.mapping)
print("Eval Inputs:   ", eval_inputs.dims.mapping)
print("Eval Targets:  ", eval_targets.dims.mapping)
print("Eval Forcings: ", eval_forcings.dims.mapping)


All Examples:   {'lat': 721, 'lon': 1440, 'batch': 1, 'time': 14, 'level': 13}
Train Inputs:   {'batch': 1, 'time': 2, 'lat': 721, 'lon': 1440, 'level': 13}
Train Targets:  {'batch': 1, 'time': 10, 'lat': 721, 'lon': 1440, 'level': 13}
Train Forcings: {'batch': 1, 'time': 10, 'lat': 721, 'lon': 1440}
Eval Inputs:    {'batch': 1, 'time': 2, 'lat': 721, 'lon': 1440, 'level': 13}
Eval Targets:   {'batch': 1, 'time': 12, 'lat': 721, 'lon': 1440, 'level': 13}
Eval Forcings:  {'batch': 1, 'time': 12, 'lat': 721, 'lon': 1440}


In [14]:
# @title Load normalization data

with gcs_bucket.blob(dir_prefix+"stats/diffs_stddev_by_level.nc").open("rb") as f:
  diffs_stddev_by_level = xarray.load_dataset(f).compute()
with gcs_bucket.blob(dir_prefix+"stats/mean_by_level.nc").open("rb") as f:
  mean_by_level = xarray.load_dataset(f).compute()
with gcs_bucket.blob(dir_prefix+"stats/stddev_by_level.nc").open("rb") as f:
  stddev_by_level = xarray.load_dataset(f).compute()

In [15]:
# @title Frozen Encoder+Processor Utilities

def _merge_params(trainable_params: dict, frozen_params: dict) -> dict:
  """Merge trainable and frozen parameter dicts back into a single param tree.

  Haiku params are two levels deep: {module_path: {param_name: array}}.
  When a module_path exists in both dicts (should not happen with correct
  partitioning), trainable values take precedence.
  """
  merged = {}
  all_keys = set(frozen_params) | set(trainable_params)
  for key in all_keys:
    if key in frozen_params and key in trainable_params:
      merged[key] = {**frozen_params[key], **trainable_params[key]}
    elif key in frozen_params:
      merged[key] = frozen_params[key]
    else:
      merged[key] = trainable_params[key]
  return merged


def partition_params_by_module(params: dict, frozen_module_names: list = None) -> tuple:
  """Partition parameters into trainable and frozen groups based on module names.

  Uses exact top-level key matching: a Haiku module path like
  'grid2mesh_gnn/~/linear' is frozen only if one of frozen_module_names
  is an exact prefix segment before the first '/'.  This avoids the
  substring bug where 'mesh_gnn' would also match 'grid2mesh_gnn'.

  Args:
    params: Full parameter tree from the model (Haiku two-level dict).
    frozen_module_names: List of Haiku module name prefixes to freeze.
                         Default: ['grid2mesh_gnn', 'mesh_gnn']
                         (encoder + processor).

  Returns:
    Tuple of (trainable_params, frozen_params).
  """
  if frozen_module_names is None:
    frozen_module_names = ['grid2mesh_gnn', 'mesh_gnn']

  trainable_params = {}
  frozen_params = {}

  def _is_frozen_key(module_path: str) -> bool:
    """Check if a top-level module path belongs to a frozen module.

    Matches the module name that appears before the first '/' separator.
    E.g. 'mesh_gnn/~/layer_0/linear' -> top-level name is 'mesh_gnn'.
    """
    top_level_name = module_path.split('/')[0]
    return top_level_name in frozen_module_names

  for module_path, sub_dict in params.items():
    if _is_frozen_key(module_path):
      frozen_params[module_path] = sub_dict
    else:
      trainable_params[module_path] = sub_dict

  return trainable_params, frozen_params


def count_params_by_module(params: dict, frozen_module_names: list = None) -> dict:
  """Count parameters in trainable and frozen modules.

  Args:
    params: Full parameter tree from the model.
    frozen_module_names: List of module names to consider as frozen.

  Returns:
    Dictionary with counts and percentages.
  """
  trainable, frozen = partition_params_by_module(params, frozen_module_names)

  def _count_leaves(tree):
    """Count total number of scalar parameters across all leaves."""
    leaves = jax.tree_util.tree_leaves(tree)
    return sum(int(np.prod(leaf.shape)) for leaf in leaves if hasattr(leaf, 'shape'))

  trainable_count = _count_leaves(trainable)
  frozen_count = _count_leaves(frozen)
  total_count = trainable_count + frozen_count

  return {
      'trainable': trainable_count,
      'frozen': frozen_count,
      'total': total_count,
      'trainable_pct': 100.0 * trainable_count / total_count if total_count > 0 else 0,
      'frozen_pct': 100.0 * frozen_count / total_count if total_count > 0 else 0,
  }


print("Frozen encoder+processor utilities loaded successfully")

Frozen encoder+processor utilities loaded successfully


In [16]:
# @title Build jitted functions with frozen encoder + processor

# No need for a GraphCastFrozen subclass: graphcast.GraphCast already has
# freeze_encoder / freeze_processor flags that insert jax.lax.stop_gradient
# at the right places in __call__.  This prevents JAX from storing encoder
# and processor activations (1 + gnn_msg_steps message-passing steps)
# during backprop, so only the decoder's single step needs gradient storage.

def _build_wrapped_predictor(model_config, task_config,
                             freeze_encoder=False, freeze_processor=False):
  """Shared builder: wraps GraphCast with casting, norm, autoregressive."""
  predictor = graphcast.GraphCast(
      model_config, task_config,
      freeze_encoder=freeze_encoder,
      freeze_processor=freeze_processor)
  predictor = casting.Bfloat16Cast(predictor)
  predictor = normalization.InputsAndResiduals(
      predictor,
      diffs_stddev_by_level=diffs_stddev_by_level,
      mean_by_level=mean_by_level,
      stddev_by_level=stddev_by_level)
  predictor = autoregressive.Predictor(predictor, gradient_checkpointing=True)
  return predictor


# Inference: no stop_gradient so XLA can fully optimize the forward pass.
@hk.transform_with_state
def run_forward(model_config, task_config, inputs, targets_template, forcings):
  predictor = _build_wrapped_predictor(model_config, task_config)
  return predictor(inputs, targets_template=targets_template, forcings=forcings)


# Training: stop_gradient on encoder+processor outputs prevents activation
# storage for all frozen steps. Only the decoder (mesh2grid_gnn) is
# differentiable, which dramatically reduces HBM usage.
@hk.transform_with_state
def loss_fn(model_config, task_config, inputs, targets, forcings):
  predictor = _build_wrapped_predictor(
      model_config, task_config,
      freeze_encoder=True, freeze_processor=True)
  loss, diagnostics = predictor.loss(inputs, targets, forcings)
  return xarray_tree.map_structure(
      lambda x: xarray_jax.unwrap_data(x.mean(), require_jax=True),
      (loss, diagnostics))


def grads_fn_with_frozen_processor(trainable_params, frozen_params, state,
                                   model_config, task_config,
                                   inputs, targets, forcings):
  """Compute gradients only w.r.t. trainable parameters (decoder only).

  Memory optimization (two complementary mechanisms):
  1. Parameter partitioning: jax.value_and_grad differentiates only w.r.t.
     trainable_params (decoder), so frozen params receive no gradient storage.
  2. stop_gradient (inside GraphCast.__call__): prevents JAX from
     backpropagating through encoder+processor, avoiding activation storage
     for all frozen message-passing steps.
  """
  def _aux(trainable_params_in):
    merged_params = _merge_params(trainable_params_in, frozen_params)
    (loss, diagnostics), next_state = loss_fn.apply(
        merged_params, state, jax.random.PRNGKey(0), model_config, task_config,
        inputs, targets, forcings)
    return loss, (diagnostics, next_state)

  (loss, (diagnostics, next_state)), grads = jax.value_and_grad(
      _aux, has_aux=True)(trainable_params)

  return loss, diagnostics, next_state, grads


def with_configs(fn):
  return functools.partial(
      fn, model_config=model_config, task_config=task_config)


init_jitted = jax.jit(with_configs(run_forward.init))

if params is None:
  params, state = init_jitted(
      rng=jax.random.PRNGKey(0),
      inputs=train_inputs,
      targets_template=train_targets,
      forcings=train_forcings)

# Partition params: freeze encoder (grid2mesh_gnn) and processor (mesh_gnn).
# Only the decoder (mesh2grid_gnn) remains trainable.
frozen_module_names = ['grid2mesh_gnn', 'mesh_gnn']
trainable_params, frozen_params = partition_params_by_module(
    params, frozen_module_names=frozen_module_names)

param_stats = count_params_by_module(params, frozen_module_names=frozen_module_names)
print("=" * 60)
print("PARAMETER STATISTICS (frozen encoder + processor)")
print("=" * 60)
print(f"Trainable parameters: {param_stats['trainable']:,} ({param_stats['trainable_pct']:.1f}%)")
print(f"Frozen parameters:    {param_stats['frozen']:,} ({param_stats['frozen_pct']:.1f}%)")
print(f"Total parameters:     {param_stats['total']:,}")
print("=" * 60)
print("Encoder (grid2mesh_gnn): FROZEN  |  run_forward: vanilla GraphCast")
print("Processor (mesh_gnn):    FROZEN  |  loss_fn: GraphCast(freeze_encoder/processor=True)")
print("Decoder (mesh2grid_gnn): TRAINABLE")
print("=" * 60)

# Jitted gradient function — accepts trainable_params, frozen_params, state
# as explicit arguments so the training loop can pass updated values each step.
grads_fn_jitted = jax.jit(with_configs(grads_fn_with_frozen_processor))


PARAMETER STATISTICS (frozen encoder + processor)
Trainable parameters: 2,979,494 (8.3%)
Frozen parameters:    33,040,384 (91.7%)
Total parameters:     36,019,878
Encoder (grid2mesh_gnn): FROZEN  |  run_forward: vanilla GraphCast
Processor (mesh_gnn):    FROZEN  |  loss_fn: GraphCast(freeze_encoder/processor=True)
Decoder (mesh2grid_gnn): TRAINABLE


# Run the model

Note that the cell below may take a while (possibly minutes) to run the first time you execute them, because this will include the time it takes for the code to compile. The second time running will be significantly faster.

This use the python loop to iterate over prediction steps, where the 1-step prediction is jitted. This has lower memory requirements than the training steps below, and should enable making prediction with the small GraphCast model on 1 deg resolution data for 4 steps.

**Note:** In this notebook, the processor weights are frozen, so predictions won't improve as much during training, but memory usage will be significantly reduced.

In [17]:
# @title Autoregressive rollout (loop in python)

assert model_config.resolution in (0, 360. / eval_inputs.sizes["lon"]), (
  "Model resolution doesn't match the data resolution. You likely want to "
  "re-filter the dataset list, and download the correct data.")

print("Inputs:  ", eval_inputs.dims.mapping)
print("Targets: ", eval_targets.dims.mapping)
print("Forcings:", eval_forcings.dims.mapping)

# Merge trainable (decoder) and frozen (encoder+processor) params for inference.
merged_params_for_eval = _merge_params(trainable_params, frozen_params)

# rollout.chunked_prediction expects fn(rng, inputs, targets_template, forcings)
# returning only predictions (not a tuple).
def run_forward_eval_wrapper(rng, inputs, targets_template, forcings):
  predictions, _ = run_forward.apply(
      merged_params_for_eval, state, rng,
      model_config, task_config,
      inputs, targets_template, forcings)
  return predictions

predictions = rollout.chunked_prediction(
    run_forward_eval_wrapper,
    rng=jax.random.PRNGKey(0),
    inputs=eval_inputs,
    targets_template=eval_targets * np.nan,
    forcings=eval_forcings)
predictions

Inputs:   {'batch': 1, 'time': 2, 'lat': 721, 'lon': 1440, 'level': 13}
Targets:  {'batch': 1, 'time': 12, 'lat': 721, 'lon': 1440, 'level': 13}
Forcings: {'batch': 1, 'time': 12, 'lat': 721, 'lon': 1440}


<xarray.Dataset> Size: 4GB
Dimensions:                  (time: 12, batch: 1, lat: 721, lon: 1440, level: 13)
Coordinates:
  * time                     (time) timedelta64[ns] 96B 0 days 06:00:00 ... 3...
  * lat                      (lat) float32 3kB -90.0 -89.75 -89.5 ... 89.75 90.0
  * lon                      (lon) float32 6kB 0.0 0.25 0.5 ... 359.5 359.8
  * level                    (level) int32 52B 50 100 150 200 ... 850 925 1000
Dimensions without coordinates: batch
Data variables:
    10m_u_component_of_wind  (time, batch, lat, lon) float32 50MB -3.845 ... ...
    10m_v_component_of_wind  (time, batch, lat, lon) float32 50MB -0.89 ... -...
    2m_temperature           (time, batch, lat, lon) float32 50MB 248.1 ... 2...
    geopotential             (time, batch, level, lat, lon) float32 648MB 1.9...
    mean_sea_level_pressure  (time, batch, lat, lon) float32 50MB 9.991e+04 ....
    specific_humidity        (time, batch, level, lat, lon) float32 648MB 3.0...
    temperature              (time, batch, level, lat, lon) float32 648MB 238...
    total_precipitation_6hr  (time, batch, lat, lon) float32 50MB 0.0002755 ....
    u_component_of_wind      (time, batch, level, lat, lon) float32 648MB 1.1...
    v_component_of_wind      (time, batch, level, lat, lon) float32 648MB -1....
    vertical_velocity        (time, batch, level, lat, lon) float32 648MB -0....

# Train the model (with Frozen Encoder + Processor)

The following operations require a large amount of memory and, depending on the accelerator being used, will only fit the very small "random" model on low resolution data. It uses the number of training steps selected above.

**KEY DIFFERENCE:** Both the encoder (grid2mesh_gnn) and the processor (mesh_gnn) weights are frozen. **Only the decoder (mesh2grid_gnn) is trained.** This significantly reduces memory usage during backpropagation because:
1. `jax.lax.stop_gradient` is inserted after encoder and processor outputs, so JAX does not store their activations for gradient computation.
2. `jax.value_and_grad` is applied only to the decoder parameter subtree, so no gradient buffers are allocated for frozen weights.

The first time executing the cell takes more time, as it includes the time to jit the function.

In [18]:
# @title Loss computation (autoregressive loss over multiple steps)

# Merge trainable (decoder) and frozen (encoder+processor) params for loss eval.
merged_params = _merge_params(trainable_params, frozen_params)

# Wrap with jit so XLA can optimize buffer reuse and gradient_checkpointing
# is effective. loss_fn.apply called bare (no jit) causes the OOM.
_loss_fn_jitted = jax.jit(
    functools.partial(loss_fn.apply, model_config=model_config, task_config=task_config))

(loss, diagnostics), next_state = _loss_fn_jitted(
    merged_params, state, jax.random.PRNGKey(0),
    inputs=train_inputs, targets=train_targets, forcings=train_forcings)
print("Loss:", float(loss))

Loss: 2.3929688930511475


In [19]:
# @title Free GPU memory before gradient computation
# Cells 15 and 16 (inference rollout + forward-only loss) leave their
# JIT memory pool allocated on the GPU. Deleting the large output arrays
# and clearing the JIT cache releases that memory before the backward pass.
import gc

for _var in ["predictions", "merged_params_for_eval", "merged_params",
             "_loss_fn_jitted"]:
    try:
        del globals()[_var]
    except KeyError:
        pass

gc.collect()
jax.clear_caches()  # evict compiled XLA binaries from host cache

print("GPU memory freed. Ready for gradient computation.")

GPU memory freed. Ready for gradient computation.


In [20]:
# @title Gradient computation (backprop through time with frozen encoder+processor)

# grads_fn_jitted requires trainable_params, frozen_params, state as explicit
# positional arguments (with_configs only bakes in model_config/task_config).
loss, diagnostics, next_state, grads = grads_fn_jitted(
    trainable_params,
    frozen_params,
    state,
    inputs=train_inputs,
    targets=train_targets,
    forcings=train_forcings)

# Compute gradient statistics.
# JAX gradients are jax.Array — use hasattr('shape') instead of isinstance(np.ndarray).
flat_grads, _ = jax.tree_util.tree_flatten(grads)
grad_norms = [float(jnp.abs(g).mean()) for g in flat_grads if hasattr(g, 'shape')]
mean_grad = np.mean(grad_norms) if grad_norms else 0.0

print("=" * 60)
print("GRADIENT STATISTICS (Frozen Encoder + Processor)")
print("=" * 60)
print(f"Loss: {loss:.6f}")
print(f"Mean |grad|: {mean_grad:.6f}")
print(f"Number of decoder gradient tensors: {len(flat_grads)}")
print("=" * 60)

# Verify that grads only contain decoder (mesh2grid_gnn) keys.
# With param partitioning, jax.value_and_grad is called only on trainable_params,
# so frozen encoder/processor keys are structurally absent from grads — they
# were never part of the differentiated pytree.
frozen_keys_in_grads = [
    k for k in grads.keys()
    if k.split('/')[0] in frozen_module_names
]
decoder_keys_in_grads = [
    k for k in grads.keys()
    if k.split('/')[0] not in frozen_module_names
]
assert len(frozen_keys_in_grads) == 0, (
    f"Frozen module keys found in grads — partitioning is broken: {frozen_keys_in_grads}")
print(f"Decoder gradient entries: {len(decoder_keys_in_grads)}")
print(f"Frozen module entries in grads: 0 (verified — encoder+processor not differentiated)")

GRADIENT STATISTICS (Frozen Encoder + Processor)
Loss: 2.383594
Mean |grad|: 0.009248
Number of decoder gradient tensors: 30
Decoder gradient entries: 15
Frozen module entries in grads: 0 (verified — encoder+processor not differentiated)


In [21]:
# @title Autoregressive rollout (keep the loop in JAX with frozen processor)

print("Inputs:  ", train_inputs.dims.mapping)
print("Targets: ", train_targets.dims.mapping)
print("Forcings:", train_forcings.dims.mapping)

# Merge trainable (decoder) and frozen (encoder+processor) params for inference.
merged_params = _merge_params(trainable_params, frozen_params)

# run_forward.apply returns (predictions, updated_state) — unpack the tuple.
predictions, _ = run_forward.apply(
    merged_params, state, jax.random.PRNGKey(0),
    model_config, task_config,
    train_inputs, train_targets * np.nan, train_forcings)
predictions

Inputs:   {'batch': 1, 'time': 2, 'lat': 721, 'lon': 1440, 'level': 13}
Targets:  {'batch': 1, 'time': 10, 'lat': 721, 'lon': 1440, 'level': 13}
Forcings: {'batch': 1, 'time': 10, 'lat': 721, 'lon': 1440}


<xarray.Dataset> Size: 3GB
Dimensions:                  (time: 10, batch: 1, lat: 721, lon: 1440, level: 13)
Coordinates:
  * time                     (time) timedelta64[ns] 80B 0 days 06:00:00 ... 2...
  * lat                      (lat) float32 3kB -90.0 -89.75 -89.5 ... 89.75 90.0
  * lon                      (lon) float32 6kB 0.0 0.25 0.5 ... 359.5 359.8
  * level                    (level) int32 52B 50 100 150 200 ... 850 925 1000
Dimensions without coordinates: batch
Data variables:
    10m_u_component_of_wind  (time, batch, lat, lon) float32 42MB xarray_jax....
    10m_v_component_of_wind  (time, batch, lat, lon) float32 42MB xarray_jax....
    2m_temperature           (time, batch, lat, lon) float32 42MB xarray_jax....
    geopotential             (time, batch, level, lat, lon) float32 540MB xar...
    mean_sea_level_pressure  (time, batch, lat, lon) float32 42MB xarray_jax....
    specific_humidity        (time, batch, level, lat, lon) float32 540MB xar...
    temperature              (time, batch, level, lat, lon) float32 540MB xar...
    total_precipitation_6hr  (time, batch, lat, lon) float32 42MB xarray_jax....
    u_component_of_wind      (time, batch, level, lat, lon) float32 540MB xar...
    v_component_of_wind      (time, batch, level, lat, lon) float32 540MB xar...
    vertical_velocity        (time, batch, level, lat, lon) float32 540MB xar...

# Example: Basic Training Loop with Frozen Processor

Below is an example of how to implement a simple training loop that respects the frozen processor weights.

In [ ]:
# jax.clear_caches()

In [22]:
import optax

# Training configuration
num_epochs = 100
learning_rate = 0.001

# Initialize optimizer (only for trainable params — decoder weights)
tx = optax.adam(learning_rate)
opt_state = tx.init(trainable_params)

# grads_fn_jitted accepts (trainable_params, frozen_params, state, ...) as
# explicit arguments, so updated params can be passed each iteration.

# Training loop
losses = []
current_trainable_params = trainable_params
current_state = state

for epoch in range(num_epochs):
    loss, diagnostics, current_state, grads = grads_fn_jitted(
        current_trainable_params,
        frozen_params,
        current_state,
        inputs=train_inputs,
        targets=train_targets,
        forcings=train_forcings)

    updates, opt_state = tx.update(grads, opt_state)
    current_trainable_params = optax.apply_updates(current_trainable_params, updates)

    losses.append(float(loss))
    print(f"Epoch {epoch+1}/{num_epochs}: Loss = {loss:.6f}")

# Update globals after training
trainable_params = current_trainable_params
state = current_state

print(f"\nTraining complete. Final loss: {losses[-1]:.6f}")
print(f"Encoder and processor weights remained frozen throughout training")

ValueError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 58933929296 bytes.